# Chapter 07: Graph Theory

**Source span:** PDF pages 65-77 of *Mathematical Foundations of Geometric Deep Learning*.

**Chapter goal:** Build graphs as discrete geometric domains: nodes carry signals, edges define locality, graph symmetries control which computations are meaningful, and Laplacian spectra turn graph signals into frequency components.

Graph theory enters geometric deep learning because a graph is more than a data structure. It is a rule for who can interact with whom. Once a graph is fixed, a node feature is a function on a discrete domain, a neighborhood is a local chart for computation, a shortest path is a geodesic in the graph metric, and a message-passing layer is a local operator constrained by the edge set.

This notebook treats the chapter as a sequence of executable inspections. Every main definition is paired with a graph, matrix, table, spectrum, or invariant check. The examples are intentionally small enough that the adjacency matrix and the drawing can be compared by eye.

## Translation guide: computational dictionary

| Book concept | Computational object | What to inspect |
| --- | --- | --- |
| Graph `G = (V, E)` | `networkx.Graph` or `networkx.DiGraph` | Whether edges are directed, weighted, self-loops, or absent |
| One-hop neighborhood `N(i)` | `list(G.neighbors(i))` | Which nodes can contribute to a one-layer update |
| Adjacency matrix `A` | `nx.to_numpy_array(G, nodelist=...)` | Sparsity, symmetry, weights, and node ordering |
| Degree matrix `D` | diagonal matrix from row sums | In/out degree for directed graphs, weighted degree for weighted graphs |
| Graph geodesic | shortest-path matrix | Distances through edges, not through the page drawing |
| Graph family | constructor such as `path_graph`, `cycle_graph`, `complete_graph` | Edge count, regularity, components, diameter, bipartiteness |
| Geometric graph | points plus a proximity rule | Which edges are induced by distances in a metric space |
| Homophily | fraction of same-label edges | Whether local averaging reinforces or mixes labels |
| Permutation | permutation matrix `P` | Relabeling changes arrays but preserves graph invariants |
| Homomorphism | vertex map preserving edges | Edges of the source still land on edges of the target |
| Laplacian `L = D - A` | symmetric positive semidefinite matrix | Smoothness energy and connected components |
| Graph Fourier transform | eigenvectors of `L` | Low eigenvalues vary slowly across edges |
| Message passing | aggregate-update loop | Information radius grows by one hop per layer |

## Visual storyboard

1. **Graph anatomy:** compare a graph with the induced neighborhood subgraph of one node.
2. **Adjacency and degree:** align a finite binary tree drawing with its adjacency and degree matrices.
3. **Graph distances and families:** compute shortest paths, diameter, regularity, and bipartiteness across common graph types.
4. **Geometric graphs and homophily:** compare unit-disk and k-nearest-neighbor edge rules, then measure same-label edge fractions.
5. **Permutation structure:** verify `A' = P A P.T`, invariant pooling, and equivariant linear message updates.
6. **Homomorphisms:** map a six-cycle into a triangle and test edge preservation.
7. **Laplacians and Fourier modes:** connect `x.T L x` to graph smoothness and decompose a signal in Laplacian eigenvectors.
8. **Message passing:** watch the nonzero support of a signal expand exactly by graph distance.

Static PNG artifacts are used for matrix and spectral comparisons because they should be readable in exported form. Plotly HTML is used for the geometric graph because hover inspection of coordinates and labels is useful. CSV and JSON artifacts record the numerical invariants that the figures invite you to check.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from matplotlib.patches import ConnectionPatch
from plotly.subplots import make_subplots
from scipy.spatial import KDTree, distance_matrix

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MFGDL book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_chapter_artifacts, assert_nonblank_image

CHAPTER = "chapter-07"
COURSE_ARTIFACTS = BOOK_ROOT / "artifacts"
ARTIFACT_ROOT = COURSE_ARTIFACTS / CHAPTER
for kind in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)
plt.rcParams.update({
    "figure.dpi": 140,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

palette = {
    "node": "#e5e7eb",
    "focus": "#f59e0b",
    "neighbor": "#38bdf8",
    "edge": "#334155",
    "soft": "#94a3b8",
    "same": "#16a34a",
    "cross": "#dc2626",
    "accent": "#7c3aed",
}

artifact_paths = []
image_paths = []
checks = {}


def track(path, *, image=False):
    path = Path(path)
    if path not in artifact_paths:
        artifact_paths.append(path)
    if image and path not in image_paths:
        image_paths.append(path)
    return path


def book_rel(path):
    path = Path(path)
    try:
        return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


def draw_matrix(ax, matrix, *, title, labels=None, cmap="Blues", vmin=None, vmax=None):
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, pad=8)
    if labels is not None:
        ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha="right")
        ax.set_yticks(range(len(labels)), labels=labels)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if abs(value) > 1e-12:
                label = f"{value:.1f}" if abs(value - round(value)) > 1e-12 else str(int(round(value)))
                ax.text(j, i, label, ha="center", va="center", color="#0f172a", fontsize=8)
    return im


def graph_homophily(G, labels):
    if G.number_of_edges() == 0:
        return math.nan
    same = sum(1 for u, v in G.edges() if labels[u] == labels[v])
    return same / G.number_of_edges()


def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, Path):
        return book_rel(value)
    if isinstance(value, float) and math.isnan(value):
        return None
    return value

BOOK_ROOT

## Graph Anatomy

A graph is a vertex set together with an edge set, but a drawing can hide the most important operational idea: for a node-level computation, the local domain is the neighborhood. The first figure highlights a focus node, its one-hop neighbors, and the induced neighborhood subgraph. This is the finite object a one-layer local rule can inspect before any multi-hop propagation happens.

Inspection target: the highlighted closed neighborhood contains the focus node and its adjacent nodes, while the induced subgraph keeps only edges whose endpoints both live in that local set.

In [ ]:
anatomy = nx.Graph()
anatomy.add_nodes_from(range(8))
anatomy.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 7), (5, 6), (6, 7)])
focus = 3
neighbors = sorted(anatomy.neighbors(focus))
closed_neighborhood = [focus, *neighbors]
neighborhood_subgraph = anatomy.subgraph(closed_neighborhood).copy()
pos = {
    0: (0.0, 1.05),
    1: (0.0, -0.15),
    2: (1.1, 0.45),
    3: (2.25, 0.45),
    4: (3.35, 1.05),
    5: (3.35, -0.15),
    6: (4.55, -0.05),
    7: (4.55, 1.0),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")

node_colors = [
    palette["focus"] if n == focus else palette["neighbor"] if n in neighbors else palette["node"]
    for n in anatomy.nodes()
]
edge_colors = [
    palette["accent"] if u in closed_neighborhood and v in closed_neighborhood else palette["soft"]
    for u, v in anatomy.edges()
]
edge_widths = [3.0 if c == palette["accent"] else 1.5 for c in edge_colors]
nx.draw_networkx_edges(anatomy, pos, ax=axes[0], edge_color=edge_colors, width=edge_widths)
nx.draw_networkx_nodes(anatomy, pos, ax=axes[0], node_color=node_colors, edgecolors="#0f172a", node_size=650)
nx.draw_networkx_labels(anatomy, pos, ax=axes[0], labels={n: f"v{n}" for n in anatomy.nodes()}, font_size=9)
axes[0].set_title("Original graph with one-hop neighborhood", pad=12)
axes[0].text(0.0, -0.62, f"N(v{focus}) = {neighbors}", fontsize=10, color="#334155")

sub_pos = {n: pos[n] for n in neighborhood_subgraph.nodes()}
sub_colors = [palette["focus"] if n == focus else palette["neighbor"] for n in neighborhood_subgraph.nodes()]
nx.draw_networkx_edges(neighborhood_subgraph, sub_pos, ax=axes[1], edge_color=palette["accent"], width=3)
nx.draw_networkx_nodes(neighborhood_subgraph, sub_pos, ax=axes[1], node_color=sub_colors, edgecolors="#0f172a", node_size=700)
nx.draw_networkx_labels(neighborhood_subgraph, sub_pos, ax=axes[1], labels={n: f"v{n}" for n in neighborhood_subgraph.nodes()}, font_size=9)
axes[1].set_title("Induced neighborhood subgraph", pad=12)
axes[1].text(2.12, -0.62, "Only local vertices and their internal edges remain", fontsize=10, color="#334155")

anatomy_path = track(
    save_matplotlib(fig, CHAPTER, "graph-anatomy-neighborhood-subgraph.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["graph_anatomy"] = {
    "focus_node": focus,
    "one_hop_neighbors": neighbors,
    "closed_neighborhood_size": len(closed_neighborhood),
    "induced_subgraph_edges": sorted([sorted(edge) for edge in neighborhood_subgraph.edges()]),
}
display_artifact(anatomy_path, width=760)

## Adjacency And Degree Matrices

The same graph has two complementary representations. A drawing makes incidence easy to see; an adjacency matrix makes algebra possible. For an undirected unweighted graph, the adjacency matrix is binary and symmetric. The diagonal degree matrix is obtained from row sums. In directed graphs, row sums and column sums separate outgoing and incoming weighted degree.

The finite binary tree below is a truncation of the level-order tree pattern from the chapter: each non-leaf has two children, and the parent-child rule appears as symmetric off-diagonal entries in `A`.

In [ ]:
tree = nx.balanced_tree(r=2, h=2)
tree_nodes = list(range(7))
tree_labels = [f"v{i + 1}" for i in tree_nodes]
tree_pos = {
    0: (0.0, 2.0),
    1: (-1.25, 1.0),
    2: (1.25, 1.0),
    3: (-1.85, 0.0),
    4: (-0.65, 0.0),
    5: (0.65, 0.0),
    6: (1.85, 0.0),
}
A_tree = nx.to_numpy_array(tree, nodelist=tree_nodes, dtype=float)
D_tree = np.diag(A_tree.sum(axis=1))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.1), gridspec_kw={"width_ratios": [1.15, 1, 1]})
axes[0].axis("off")
axes[0].set_aspect("equal")
nx.draw_networkx_edges(tree, tree_pos, ax=axes[0], edge_color=palette["edge"], width=2)
nx.draw_networkx_nodes(tree, tree_pos, ax=axes[0], node_color="#dbeafe", edgecolors="#1e3a8a", node_size=700)
nx.draw_networkx_labels(tree, tree_pos, ax=axes[0], labels={n: tree_labels[n] for n in tree_nodes}, font_size=9)
axes[0].set_title("Finite binary tree", pad=10)
draw_matrix(axes[1], A_tree, title="Adjacency A", labels=tree_labels, cmap="YlGnBu", vmin=0, vmax=1)
draw_matrix(axes[2], D_tree, title="Degree matrix D", labels=tree_labels, cmap="Oranges", vmin=0, vmax=max(np.diag(D_tree)))
fig.tight_layout()

matrix_path = track(
    save_matplotlib(fig, CHAPTER, "adjacency-degree-matrices-binary-tree.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

directed = nx.DiGraph()
directed.add_weighted_edges_from([
    ("a", "b", 1.5),
    ("a", "c", 2.0),
    ("b", "c", 0.75),
    ("c", "d", 1.25),
    ("d", "b", 0.5),
])
directed_nodes = ["a", "b", "c", "d"]
A_directed = nx.to_numpy_array(directed, nodelist=directed_nodes, weight="weight", dtype=float)
degree_rows = []
for i, node in enumerate(directed_nodes):
    degree_rows.append({
        "node": node,
        "weighted_out_degree_row_sum": float(A_directed[i, :].sum()),
        "weighted_in_degree_column_sum": float(A_directed[:, i].sum()),
    })
degree_table_path = track(save_table_csv(degree_rows, CHAPTER, "directed-weighted-degree-table.csv", root=COURSE_ARTIFACTS))

checks["adjacency_degree"] = {
    "tree_adjacency_symmetric": bool(np.allclose(A_tree, A_tree.T)),
    "tree_row_sums_equal_degrees": bool(np.allclose(A_tree.sum(axis=1), np.diag(D_tree))),
    "tree_edge_count": int(tree.number_of_edges()),
    "directed_adjacency_symmetric": bool(np.allclose(A_directed, A_directed.T)),
    "directed_total_in_equals_total_out": bool(np.isclose(A_directed.sum(axis=0).sum(), A_directed.sum(axis=1).sum())),
}

display_artifact(matrix_path, width=820)
display_artifact(degree_table_path)

## Graph Geodesics And Graph Families

A graph drawing has Euclidean distances on the page, but the graph geodesic is measured through edge weights. The weighted graph below highlights the shortest path between two vertices and places the all-pairs distance matrix next to it. The maximum finite entry is the diameter when the graph is connected.

The family dashboard then compares point-cloud/null, path, cycle, complete, bipartite, and Petersen graphs. The point is not just taxonomy: edge rules change connectedness, degree sequence, shortest paths, and what a local message-passing layer can reach.

In [ ]:
weighted = nx.Graph()
weighted.add_weighted_edges_from([
    ("v1", "v2", 2.0),
    ("v1", "v3", 4.2),
    ("v2", "v3", 1.0),
    ("v2", "v4", 5.0),
    ("v3", "v4", 2.5),
    ("v4", "v5", 1.0),
])
weighted_nodes = ["v1", "v2", "v3", "v4", "v5"]
weighted_pos = {"v1": (0, 0.8), "v2": (1.2, 1.45), "v3": (1.4, 0.2), "v4": (2.8, 0.55), "v5": (4.0, 0.55)}
shortest_path = nx.shortest_path(weighted, "v1", "v5", weight="weight")
shortest_edges = {tuple(sorted(edge)) for edge in zip(shortest_path[:-1], shortest_path[1:])}
dist_matrix = nx.floyd_warshall_numpy(weighted, nodelist=weighted_nodes, weight="weight")
diameter_weighted = float(np.max(dist_matrix[np.isfinite(dist_matrix)]))

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
axes[0].axis("off")
axes[0].set_aspect("equal")
edge_colors = [palette["accent"] if tuple(sorted((u, v))) in shortest_edges else palette["soft"] for u, v in weighted.edges()]
edge_widths = [3.5 if c == palette["accent"] else 1.8 for c in edge_colors]
nx.draw_networkx_edges(weighted, weighted_pos, ax=axes[0], edge_color=edge_colors, width=edge_widths)
nx.draw_networkx_nodes(weighted, weighted_pos, ax=axes[0], node_color="#ede9fe", edgecolors="#4c1d95", node_size=720)
nx.draw_networkx_labels(weighted, weighted_pos, ax=axes[0], font_size=9)
nx.draw_networkx_edge_labels(
    weighted,
    weighted_pos,
    ax=axes[0],
    edge_labels={(u, v): data["weight"] for u, v, data in weighted.edges(data=True)},
    font_size=8,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.75),
)
axes[0].set_title("Weighted graph with v1-to-v5 geodesic", pad=10)
axes[0].text(0, -0.15, "Shortest path: " + " -> ".join(shortest_path), color="#334155", fontsize=10)
draw_matrix(axes[1], np.asarray(dist_matrix), title="All-pairs graph distances", labels=weighted_nodes, cmap="PuBu")
axes[1].set_xlabel(f"Diameter = {diameter_weighted:.1f}")
fig.tight_layout()

distance_path = track(
    save_matplotlib(fig, CHAPTER, "graph-geodesic-distance-matrix.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

families = {
    "Null N6": nx.empty_graph(6),
    "Path P6": nx.path_graph(6),
    "Cycle C6": nx.cycle_graph(6),
    "Complete K6": nx.complete_graph(6),
    "Bipartite K3,3": nx.complete_bipartite_graph(3, 3),
    "Petersen": nx.petersen_graph(),
}

family_rows = []
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, G) in zip(axes.flat, families.items()):
    ax.axis("off")
    ax.set_aspect("equal")
    if name.startswith("Bipartite"):
        pos_family = nx.bipartite_layout(G, nodes=[0, 1, 2])
    elif name == "Petersen":
        pos_family = nx.shell_layout(G, nlist=[range(5), range(5, 10)])
    else:
        pos_family = nx.circular_layout(G)
    degree_sequence = sorted([degree for _, degree in G.degree()])
    connected = nx.is_connected(G) if G.number_of_nodes() else False
    diameter = nx.diameter(G) if connected and G.number_of_nodes() > 1 else None
    regular_degree = degree_sequence[0] if degree_sequence and len(set(degree_sequence)) == 1 else None
    family_rows.append({
        "graph_family": name,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "components": nx.number_connected_components(G),
        "connected": connected,
        "diameter_if_connected": "" if diameter is None else diameter,
        "degree_sequence": " ".join(map(str, degree_sequence)),
        "regular_degree": "" if regular_degree is None else regular_degree,
        "bipartite": nx.is_bipartite(G),
    })
    node_color = "#f8fafc" if G.number_of_edges() else "#fee2e2"
    nx.draw_networkx_edges(G, pos_family, ax=ax, edge_color=palette["soft"], width=1.6)
    nx.draw_networkx_nodes(G, pos_family, ax=ax, node_color=node_color, edgecolors="#0f172a", node_size=360)
    nx.draw_networkx_labels(G, pos_family, ax=ax, font_size=7)
    regular_label = f"{regular_degree}-regular" if regular_degree is not None else "not regular"
    diameter_label = "diameter n/a" if diameter is None else f"diameter {diameter}"
    ax.set_title(f"{name}\n{G.number_of_edges()} edges, {regular_label}, {diameter_label}", fontsize=9)
fig.tight_layout()

family_path = track(
    save_matplotlib(fig, CHAPTER, "graph-family-taxonomy-diagnostics.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)
family_table_path = track(save_table_csv(family_rows, CHAPTER, "graph-family-diagnostics.csv", root=COURSE_ARTIFACTS))

checks["distances_and_families"] = {
    "weighted_shortest_path_v1_v5": shortest_path,
    "weighted_distance_v1_v5": float(dist_matrix[0, -1]),
    "weighted_diameter": diameter_weighted,
    "distance_matrix_symmetric": bool(np.allclose(dist_matrix, dist_matrix.T)),
    "complete_graph_edge_count_correct": families["Complete K6"].number_of_edges() == 6 * 5 // 2,
    "path_graph_diameter": nx.diameter(families["Path P6"]),
    "cycle_graph_regular_degree": dict(families["Cycle C6"].degree())[0],
}

display_artifact(distance_path, width=820)
display_artifact(family_path, width=820)
display_artifact(family_table_path)

## Geometric Graphs And Homophily

In a geometric graph, the vertices also have positions in a metric space. The edge set is then induced by a proximity rule rather than given directly. A unit-disk rule asks for all pairs within a radius. A k-nearest-neighbor rule asks each node to choose a fixed number of closest nodes. These two rules can produce different degrees, connected components, and long edges.

Node labels add another layer. Homophily measures how often an edge connects nodes of the same class. This matters for GNNs because local averaging is helpful when labels or features are smooth across edges, and damaging when most edges cross between classes.

In [ ]:
cluster_left = rng.normal(loc=(-1.0, 0.0), scale=(0.22, 0.27), size=(10, 2))
cluster_right = rng.normal(loc=(1.0, 0.12), scale=(0.24, 0.27), size=(10, 2))
points = np.vstack([cluster_left, cluster_right])
node_labels = np.array([0] * len(cluster_left) + [1] * len(cluster_right))
point_nodes = list(range(len(points)))
point_distances = distance_matrix(points, points)

epsilon = 0.58
unit_disk = nx.Graph()
unit_disk.add_nodes_from(point_nodes)
for i, j in itertools.combinations(point_nodes, 2):
    if point_distances[i, j] <= epsilon:
        unit_disk.add_edge(i, j, weight=float(point_distances[i, j]))

k = 3
tree = KDTree(points)
_, neighbor_indices = tree.query(points, k=k + 1)
knn_directed_edges = []
knn_graph = nx.Graph()
knn_graph.add_nodes_from(point_nodes)
for i, row in enumerate(neighbor_indices):
    for j in row[1:]:
        knn_directed_edges.append((i, int(j)))
        knn_graph.add_edge(i, int(j), weight=float(point_distances[i, int(j)]))


def edge_trace(G):
    x_values, y_values = [], []
    for u, v in G.edges():
        x_values.extend([points[u, 0], points[v, 0], None])
        y_values.extend([points[u, 1], points[v, 1], None])
    return go.Scatter(x=x_values, y=y_values, mode="lines", line=dict(color="#64748b", width=1.5), hoverinfo="skip")


def node_trace(title):
    colors = ["#2563eb" if label == 0 else "#f97316" for label in node_labels]
    return go.Scatter(
        x=points[:, 0],
        y=points[:, 1],
        mode="markers+text",
        text=[str(i) for i in point_nodes],
        textposition="top center",
        marker=dict(size=12, color=colors, line=dict(color="#0f172a", width=1)),
        hovertext=[f"node {i}<br>label {node_labels[i]}<br>x={points[i,0]:.2f}, y={points[i,1]:.2f}" for i in point_nodes],
        hoverinfo="text",
        name=title,
    )


fig_plotly = make_subplots(rows=1, cols=2, subplot_titles=(f"Unit disk: radius {epsilon}", f"Symmetric kNN: k={k}"))
fig_plotly.add_trace(edge_trace(unit_disk), row=1, col=1)
fig_plotly.add_trace(node_trace("unit disk nodes"), row=1, col=1)
fig_plotly.add_trace(edge_trace(knn_graph), row=1, col=2)
fig_plotly.add_trace(node_trace("kNN nodes"), row=1, col=2)
fig_plotly.update_xaxes(scaleanchor="y", scaleratio=1, showgrid=True, zeroline=False)
fig_plotly.update_yaxes(showgrid=True, zeroline=False)
fig_plotly.update_layout(height=520, width=920, showlegend=False, title="Geometric graph edge rules on the same point cloud")
geo_html_path = track(save_plotly_html(fig_plotly, CHAPTER, "geometric-graphs-unitdisk-knn.html", root=COURSE_ARTIFACTS))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.7))
for ax, G, title in [
    (axes[0], unit_disk, f"Unit disk, epsilon={epsilon}"),
    (axes[1], knn_graph, f"Symmetric kNN, k={k}"),
]:
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for u, v in G.edges():
        ax.plot([points[u, 0], points[v, 0]], [points[u, 1], points[v, 1]], color="#94a3b8", lw=1.3, zorder=1)
    ax.scatter(points[node_labels == 0, 0], points[node_labels == 0, 1], s=80, c="#2563eb", edgecolors="#0f172a", label="class 0", zorder=2)
    ax.scatter(points[node_labels == 1, 0], points[node_labels == 1, 1], s=80, c="#f97316", edgecolors="#0f172a", label="class 1", zorder=2)
    for i, (x, y) in enumerate(points):
        ax.text(x + 0.025, y + 0.025, str(i), fontsize=7)
    ax.text(
        0.02,
        0.02,
        f"edges={G.number_of_edges()}\ncomponents={nx.number_connected_components(G)}\nhomophily={graph_homophily(G, node_labels):.2f}",
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
    )
axes[1].legend(loc="upper right", frameon=False)
fig.tight_layout()
geo_png_path = track(
    save_matplotlib(fig, CHAPTER, "geometric-graphs-proximity-rules.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

homophilic = nx.Graph()
homophilic.add_nodes_from(point_nodes)
for label_value in [0, 1]:
    group = [i for i in point_nodes if node_labels[i] == label_value]
    for i, j in itertools.combinations(group, 2):
        if point_distances[i, j] <= 0.62:
            homophilic.add_edge(i, j)

heterophilic = nx.Graph()
heterophilic.add_nodes_from(point_nodes)
for i in point_nodes:
    opposite = [j for j in point_nodes if node_labels[j] != node_labels[i]]
    nearest_opposite = min(opposite, key=lambda j: point_distances[i, j])
    heterophilic.add_edge(i, nearest_opposite)

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.7))
for ax, G, title in [
    (axes[0], homophilic, "Mostly same-label edges"),
    (axes[1], heterophilic, "Forced cross-label edges"),
]:
    ax.set_aspect("equal")
    ax.set_title(f"{title}\nhomophily h={graph_homophily(G, node_labels):.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
    for u, v in G.edges():
        color = palette["same"] if node_labels[u] == node_labels[v] else palette["cross"]
        ax.plot([points[u, 0], points[v, 0]], [points[u, 1], points[v, 1]], color=color, lw=1.6, alpha=0.85)
    ax.scatter(points[node_labels == 0, 0], points[node_labels == 0, 1], s=80, c="#2563eb", edgecolors="#0f172a", zorder=2)
    ax.scatter(points[node_labels == 1, 0], points[node_labels == 1, 1], s=80, c="#f97316", edgecolors="#0f172a", zorder=2)
fig.tight_layout()
homophily_path = track(
    save_matplotlib(fig, CHAPTER, "homophily-edge-label-comparison.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["geometric_graphs"] = {
    "unit_disk_edges": unit_disk.number_of_edges(),
    "unit_disk_components": nx.number_connected_components(unit_disk),
    "unit_disk_all_edges_within_epsilon": all(point_distances[u, v] <= epsilon + 1e-12 for u, v in unit_disk.edges()),
    "knn_directed_out_degree": k,
    "knn_graph_edges": knn_graph.number_of_edges(),
    "homophilic_h": float(graph_homophily(homophilic, node_labels)),
    "heterophilic_h": float(graph_homophily(heterophilic, node_labels)),
    "homophilic_exceeds_heterophilic": graph_homophily(homophilic, node_labels) > graph_homophily(heterophilic, node_labels),
}

display_artifact(geo_png_path, width=820)
display_artifact(geo_html_path, width="100%", height=560)
display_artifact(homophily_path, width=820)

## Applied lab: Thresholds Change The Graph Domain

The same point cloud can support many graph domains. In this lab, only the unit-disk radius changes. Watch the edge count, number of components, homophily, and diameter. A small radius can fragment the graph; a large radius can connect classes so aggressively that the graph prior stops matching the label geometry.

In [ ]:
threshold_rows = []
threshold_values = np.round(np.linspace(0.22, 1.75, 16), 3)
for eps in threshold_values:
    G_eps = nx.Graph()
    G_eps.add_nodes_from(point_nodes)
    for i, j in itertools.combinations(point_nodes, 2):
        if point_distances[i, j] <= eps:
            G_eps.add_edge(i, j)
    connected = nx.is_connected(G_eps)
    threshold_rows.append({
        "epsilon": float(eps),
        "edges": G_eps.number_of_edges(),
        "components": nx.number_connected_components(G_eps),
        "largest_component_size": len(max(nx.connected_components(G_eps), key=len)),
        "mean_degree": float(np.mean([degree for _, degree in G_eps.degree()])),
        "homophily": None if G_eps.number_of_edges() == 0 else float(graph_homophily(G_eps, node_labels)),
        "diameter_if_connected": None if not connected else nx.diameter(G_eps),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_table_path = track(save_table_csv(threshold_rows, CHAPTER, "geometric-threshold-sweep.csv", root=COURSE_ARTIFACTS))

fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.2), sharex=True)
axes = axes.ravel()
axes[0].plot(threshold_df["epsilon"], threshold_df["edges"], marker="o", color="#2563eb")
axes[0].set_title("Edge count")
axes[1].plot(threshold_df["epsilon"], threshold_df["components"], marker="o", color="#dc2626")
axes[1].set_title("Connected components")
axes[2].plot(threshold_df["epsilon"], threshold_df["homophily"], marker="o", color="#16a34a")
axes[2].set_title("Homophily")
axes[2].set_ylim(-0.05, 1.05)
axes[3].plot(threshold_df["epsilon"], threshold_df["diameter_if_connected"], marker="o", color="#7c3aed")
axes[3].set_title("Diameter once connected")
for ax in axes:
    ax.set_xlabel("unit-disk radius epsilon")
    ax.grid(alpha=0.25)
fig.tight_layout()
threshold_path = track(
    save_matplotlib(fig, CHAPTER, "geometric-threshold-sweep-lab.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["threshold_lab"] = {
    "edge_count_monotone": bool(np.all(np.diff(threshold_df["edges"].to_numpy()) >= 0)),
    "components_nonincreasing": bool(np.all(np.diff(threshold_df["components"].to_numpy()) <= 0)),
    "first_connected_epsilon": float(threshold_df.loc[threshold_df["components"].eq(1), "epsilon"].iloc[0]),
}

display_artifact(threshold_path, width=760)
display_artifact(threshold_table_path)

## Permutations: Invariance And Equivariance

Node labels are bookkeeping, not geometry. If a graph is relabeled by a permutation matrix `P`, its adjacency matrix changes to `P A P.T`, but intrinsic structure remains. Graph-level pooling by sum, mean, or max is invariant to row order. Node-level linear message updates are equivariant: relabel first and then update gives the same result as update first and then relabel.

In [ ]:
perm_graph = nx.cycle_graph(5)
perm_graph.add_edge(0, 2)
perm_nodes = list(range(5))
perm_order = [2, 4, 1, 3, 0]
P = np.eye(len(perm_nodes))[perm_order]
A_perm = nx.to_numpy_array(perm_graph, nodelist=perm_nodes)
A_reordered = P @ A_perm @ P.T
A_direct_order = nx.to_numpy_array(perm_graph, nodelist=perm_order)

X = np.array([
    [1.0, 0.2],
    [0.5, 1.1],
    [1.4, -0.3],
    [-0.4, 0.7],
    [0.9, 1.6],
])
PX = P @ X
pool_sum_invariant = np.allclose(X.sum(axis=0), PX.sum(axis=0))
pool_mean_invariant = np.allclose(X.mean(axis=0), PX.mean(axis=0))
pool_max_invariant = np.allclose(X.max(axis=0), PX.max(axis=0))

A_loop = A_perm + np.eye(A_perm.shape[0])
D_loop_inv = np.diag(1.0 / A_loop.sum(axis=1))
S = D_loop_inv @ A_loop
equivariance_residual = np.linalg.norm((P @ S @ P.T) @ (P @ X) - P @ (S @ X))

pos_original = nx.circular_layout(perm_graph)
pos_reordered = {node: pos_original[node] for node in perm_graph.nodes()}
fig, axes = plt.subplots(2, 2, figsize=(10.6, 8.2))
for ax in axes.flat:
    ax.set_aspect("equal")
axes[0, 0].axis("off")
nx.draw_networkx_edges(perm_graph, pos_original, ax=axes[0, 0], edge_color=palette["soft"], width=2)
nx.draw_networkx_nodes(perm_graph, pos_original, ax=axes[0, 0], node_color="#dbeafe", edgecolors="#1e3a8a", node_size=720)
nx.draw_networkx_labels(perm_graph, pos_original, ax=axes[0, 0], labels={n: f"v{n}" for n in perm_nodes})
axes[0, 0].set_title("Original labels")

axes[0, 1].axis("off")
nx.draw_networkx_edges(perm_graph, pos_reordered, ax=axes[0, 1], edge_color=palette["soft"], width=2)
nx.draw_networkx_nodes(perm_graph, pos_reordered, ax=axes[0, 1], node_color="#ede9fe", edgecolors="#4c1d95", node_size=720)
nx.draw_networkx_labels(perm_graph, pos_reordered, ax=axes[0, 1], labels={node: f"row {perm_order.index(node)}\nold v{node}" for node in perm_nodes}, font_size=8)
axes[0, 1].set_title("Same graph in permuted row order")

draw_matrix(axes[1, 0], A_perm, title="A in original order", labels=[f"v{i}" for i in perm_nodes], cmap="YlGnBu", vmin=0, vmax=1)
draw_matrix(axes[1, 1], A_reordered, title="P A P.T", labels=[f"v{i}" for i in perm_order], cmap="YlGnBu", vmin=0, vmax=1)
fig.tight_layout()
permutation_path = track(
    save_matplotlib(fig, CHAPTER, "permutation-relabeling-equivariance.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["permutation"] = {
    "permutation_order": perm_order,
    "orthogonal": bool(np.allclose(P @ P.T, np.eye(P.shape[0]))),
    "papT_matches_direct_reordered_adjacency": bool(np.allclose(A_reordered, A_direct_order)),
    "adjacency_spectrum_invariant": bool(np.allclose(np.sort(np.linalg.eigvalsh(A_perm)), np.sort(np.linalg.eigvalsh(A_reordered)))),
    "sum_pool_invariant": bool(pool_sum_invariant),
    "mean_pool_invariant": bool(pool_mean_invariant),
    "max_pool_invariant": bool(pool_max_invariant),
    "message_update_equivariance_residual": float(equivariance_residual),
}

display_artifact(permutation_path, width=780)

## Homomorphisms

A graph homomorphism is weaker than an isomorphism. It may collapse several source vertices onto one target vertex, but it must preserve adjacency: every source edge must land on a target edge. The figure maps a six-cycle to a triangle by repeating target colors. It is not bijective, yet every cycle edge still maps to an edge of `K3`.

In [ ]:
C6 = nx.cycle_graph(6)
K3 = nx.complete_graph(3)
hom_map = {i: i % 3 for i in C6.nodes()}
target_colors = {0: "#2563eb", 1: "#f97316", 2: "#16a34a"}
edge_preserved = all(K3.has_edge(hom_map[u], hom_map[v]) for u, v in C6.edges())
is_bijective = len(set(hom_map.values())) == C6.number_of_nodes()

pos_c6 = nx.circular_layout(C6)
pos_k3 = nx.circular_layout(K3)
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.8))
for ax in axes:
    ax.axis("off")
    ax.set_aspect("equal")

nx.draw_networkx_edges(C6, pos_c6, ax=axes[0], edge_color=palette["soft"], width=2)
nx.draw_networkx_nodes(C6, pos_c6, ax=axes[0], node_color=[target_colors[hom_map[n]] for n in C6.nodes()], edgecolors="#0f172a", node_size=760)
nx.draw_networkx_labels(C6, pos_c6, ax=axes[0], labels={n: f"{n}->{hom_map[n]}" for n in C6.nodes()}, font_size=8)
axes[0].set_title("Source C6 with target colors")

nx.draw_networkx_edges(K3, pos_k3, ax=axes[1], edge_color=palette["edge"], width=2.5)
nx.draw_networkx_nodes(K3, pos_k3, ax=axes[1], node_color=[target_colors[n] for n in K3.nodes()], edgecolors="#0f172a", node_size=860)
nx.draw_networkx_labels(K3, pos_k3, ax=axes[1], labels={n: f"K3:{n}" for n in K3.nodes()}, font_size=9)
axes[1].set_title("Target K3")

for source_node, target_node in hom_map.items():
    con = ConnectionPatch(
        xyA=pos_c6[source_node],
        coordsA=axes[0].transData,
        xyB=pos_k3[target_node],
        coordsB=axes[1].transData,
        arrowstyle="->",
        lw=1.0,
        color=target_colors[target_node],
        alpha=0.45,
    )
    fig.add_artist(con)
fig.suptitle("A non-bijective edge-preserving map", y=1.02)
fig.tight_layout()

homomorphism_path = track(
    save_matplotlib(fig, CHAPTER, "homomorphism-cycle-to-triangle.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["homomorphism"] = {
    "source": "C6",
    "target": "K3",
    "vertex_map": hom_map,
    "edge_preserved": bool(edge_preserved),
    "bijective": bool(is_bijective),
    "collapsed_vertices": C6.number_of_nodes() - len(set(hom_map.values())),
}

display_artifact(homomorphism_path, width=760)

## Laplacians And Smoothness Energy

For an undirected weighted graph, `L = D - A` is a discrete smoothness operator. A signal has low Dirichlet energy when adjacent nodes have similar values and high energy when it jumps across many edges. The same matrix also detects connected components: the number of zero Laplacian eigenvalues equals the number of components.

This example uses a barbell graph so the bridge edge is visually important. A smooth signal changes slowly from one clique to the other; a noisy signal changes sharply across many edges.

In [ ]:
lap_graph = nx.barbell_graph(5, 1)
lap_nodes = list(lap_graph.nodes())
lap_pos = nx.spring_layout(lap_graph, seed=11)
A_lap = nx.to_numpy_array(lap_graph, nodelist=lap_nodes, dtype=float)
D_lap = np.diag(A_lap.sum(axis=1))
L_lap = D_lap - A_lap
degrees = A_lap.sum(axis=1)
D_inv_sqrt = np.diag(np.where(degrees > 0, 1.0 / np.sqrt(degrees), 0.0))
L_norm = np.eye(len(lap_nodes)) - D_inv_sqrt @ A_lap @ D_inv_sqrt

smooth_by_node = {node: 1.0 if node < 5 else 0.0 if node == 5 else -1.0 for node in lap_nodes}
smooth_signal = np.array([smooth_by_node[node] for node in lap_nodes])
noisy_signal = rng.normal(size=len(lap_nodes))
energy_smooth = float(smooth_signal @ L_lap @ smooth_signal)
energy_noisy = float(noisy_signal @ L_lap @ noisy_signal)
edge_energy_smooth = float(sum((smooth_by_node[u] - smooth_by_node[v]) ** 2 for u, v in lap_graph.edges()))
evals_lap = np.linalg.eigvalsh(L_lap)
evals_norm = np.linalg.eigvalsh(L_norm)
component_count = nx.number_connected_components(lap_graph)

fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.2))
axes[0, 0].axis("off")
axes[0, 0].set_aspect("equal")
nx.draw_networkx_edges(lap_graph, lap_pos, ax=axes[0, 0], edge_color=palette["soft"], width=1.8)
nodes_drawn = nx.draw_networkx_nodes(
    lap_graph,
    lap_pos,
    ax=axes[0, 0],
    node_color=smooth_signal,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    edgecolors="#0f172a",
    node_size=520,
)
nx.draw_networkx_labels(lap_graph, lap_pos, ax=axes[0, 0], font_size=7)
axes[0, 0].set_title("Smooth signal on a barbell graph")
fig.colorbar(nodes_drawn, ax=axes[0, 0], fraction=0.046, pad=0.02)

draw_matrix(axes[0, 1], L_lap, title="Combinatorial Laplacian L", labels=[str(n) for n in lap_nodes], cmap="RdBu_r")
axes[1, 0].bar(["smooth", "noisy"], [energy_smooth, energy_noisy], color=["#16a34a", "#dc2626"])
axes[1, 0].set_title("Dirichlet energy x.T L x")
axes[1, 0].set_ylabel("energy")
axes[1, 0].grid(axis="y", alpha=0.25)
axes[1, 1].plot(np.arange(len(evals_norm)), evals_norm, marker="o", color="#7c3aed")
axes[1, 1].axhline(0, color="#0f172a", lw=0.8)
axes[1, 1].axhline(2, color="#0f172a", lw=0.8, ls="--")
axes[1, 1].set_title("Normalized Laplacian spectrum")
axes[1, 1].set_xlabel("eigenvalue index")
axes[1, 1].set_ylabel("lambda")
axes[1, 1].grid(alpha=0.25)
fig.tight_layout()

laplacian_path = track(
    save_matplotlib(fig, CHAPTER, "laplacian-dirichlet-energy.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

checks["laplacian"] = {
    "laplacian_symmetric": bool(np.allclose(L_lap, L_lap.T)),
    "laplacian_psd_min_eigenvalue": float(evals_lap.min()),
    "zero_eigenvalue_multiplicity": int(np.sum(np.isclose(evals_lap, 0.0, atol=1e-10))),
    "connected_components": int(component_count),
    "smooth_energy": energy_smooth,
    "noisy_energy": energy_noisy,
    "energy_matches_edge_sum_residual": float(abs(energy_smooth - edge_energy_smooth)),
    "normalized_spectrum_min": float(evals_norm.min()),
    "normalized_spectrum_max": float(evals_norm.max()),
}

display_artifact(laplacian_path, width=820)

## Graph Fourier Modes

Laplacian eigenvectors are graph Fourier modes. Low eigenvalues correspond to signals that vary slowly across edges; high eigenvalues oscillate rapidly. On a cycle graph, the modes resemble discrete sine and cosine waves, but the eigenvector basis inside a repeated eigenspace is not unique. The invariant facts are orthonormality, reconstruction, and the energy identity `f.T L f = sum_k lambda_k * fhat_k^2`.

In [ ]:
fourier_graph = nx.cycle_graph(16)
fourier_nodes = list(fourier_graph.nodes())
theta = np.linspace(0, 2 * np.pi, len(fourier_nodes), endpoint=False)
fourier_pos = {i: (np.cos(theta[i]), np.sin(theta[i])) for i in fourier_nodes}
A_fourier = nx.to_numpy_array(fourier_graph, nodelist=fourier_nodes, dtype=float)
L_fourier = np.diag(A_fourier.sum(axis=1)) - A_fourier
evals_fourier, U_fourier = np.linalg.eigh(L_fourier)
order = np.argsort(evals_fourier)
evals_fourier = evals_fourier[order]
U_fourier = U_fourier[:, order]

signal = 0.9 * np.cos(theta - 0.5) + 0.35 * np.sin(3 * theta) + np.where((theta > 3.6) & (theta < 4.7), 0.9, 0.0)
coefficients = U_fourier.T @ signal
reconstruction = U_fourier @ coefficients
lowpass = U_fourier[:, :5] @ coefficients[:5]
energy_direct = float(signal @ L_fourier @ signal)
energy_spectral = float(np.sum(evals_fourier * coefficients ** 2))

fig = plt.figure(figsize=(12.2, 8.6))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 0.95])
mode_indices = [0, 1, 2, 5]
for col, mode_index in enumerate(mode_indices):
    ax = fig.add_subplot(gs[0, col])
    ax.axis("off")
    ax.set_aspect("equal")
    values = U_fourier[:, mode_index]
    nx.draw_networkx_edges(fourier_graph, fourier_pos, ax=ax, edge_color="#cbd5e1", width=1.5)
    drawn = nx.draw_networkx_nodes(
        fourier_graph,
        fourier_pos,
        ax=ax,
        node_color=values,
        cmap="coolwarm",
        edgecolors="#0f172a",
        node_size=420,
    )
    ax.set_title(f"mode {mode_index}\nlambda={evals_fourier[mode_index]:.3f}", fontsize=9)
    fig.colorbar(drawn, ax=ax, fraction=0.046, pad=0.02)

ax_spectrum = fig.add_subplot(gs[1, 0])
ax_spectrum.plot(evals_fourier, marker="o", color="#7c3aed")
ax_spectrum.set_title("Cycle Laplacian spectrum")
ax_spectrum.set_xlabel("mode index")
ax_spectrum.set_ylabel("lambda")
ax_spectrum.grid(alpha=0.25)

ax_coeff = fig.add_subplot(gs[1, 1])
ax_coeff.bar(np.arange(len(coefficients)), np.abs(coefficients), color="#2563eb")
ax_coeff.set_title("Graph Fourier magnitudes")
ax_coeff.set_xlabel("mode index")
ax_coeff.set_ylabel("|fhat|")
ax_coeff.grid(axis="y", alpha=0.25)

ax_signal = fig.add_subplot(gs[1, 2:])
ax_signal.plot(fourier_nodes, signal, marker="o", label="signal", color="#0f172a")
ax_signal.plot(fourier_nodes, lowpass, marker="o", label="5-mode low-pass", color="#16a34a")
ax_signal.set_title("Signal and low-pass reconstruction")
ax_signal.set_xlabel("cycle node")
ax_signal.grid(alpha=0.25)
ax_signal.legend(frameon=False)
fig.tight_layout()

fourier_path = track(
    save_matplotlib(fig, CHAPTER, "graph-fourier-modes-and-filtering.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

fourier_rows = [
    {"mode": int(i), "eigenvalue": float(evals_fourier[i]), "coefficient": float(coefficients[i]), "abs_coefficient": float(abs(coefficients[i]))}
    for i in range(len(fourier_nodes))
]
fourier_table_path = track(save_table_csv(fourier_rows, CHAPTER, "graph-fourier-coefficients.csv", root=COURSE_ARTIFACTS))

checks["graph_fourier"] = {
    "orthonormal_residual": float(np.linalg.norm(U_fourier.T @ U_fourier - np.eye(len(fourier_nodes)))),
    "reconstruction_residual": float(np.linalg.norm(reconstruction - signal)),
    "energy_identity_residual": float(abs(energy_direct - energy_spectral)),
    "lowest_eigenvalue": float(evals_fourier[0]),
    "lowpass_error_norm": float(np.linalg.norm(lowpass - signal)),
}

display_artifact(fourier_path, width=850)
display_artifact(fourier_table_path)

## Message Passing

A message-passing layer uses a permutation-invariant aggregation over neighbors and then updates each node feature. Locality is the geometric constraint: after one layer, a source node can affect only its one-hop neighborhood; after two layers, only nodes within graph distance two; and so on.

The next experiment starts with a delta signal on one grid node and repeatedly applies a mean aggregator with self-loops. The color support expands exactly by graph distance, which is the operational meaning of the neighborhood definition.

In [ ]:
grid = nx.grid_2d_graph(4, 4)
grid = nx.convert_node_labels_to_integers(grid, ordering="sorted")
grid_nodes = list(grid.nodes())
grid_pos = {node: (node % 4, 3 - node // 4) for node in grid_nodes}
source_node = 5
A_grid = nx.to_numpy_array(grid, nodelist=grid_nodes, dtype=float)
A_grid_loop = A_grid + np.eye(len(grid_nodes))
M_grid = np.diag(1.0 / A_grid_loop.sum(axis=1)) @ A_grid_loop
x = np.zeros(len(grid_nodes))
x[source_node] = 1.0
signals = [x.copy()]
for _ in range(3):
    x = M_grid @ x
    signals.append(x.copy())

fig, axes = plt.subplots(1, 4, figsize=(13.4, 3.8))
for layer, ax in enumerate(axes):
    ax.axis("off")
    ax.set_aspect("equal")
    values = signals[layer]
    nx.draw_networkx_edges(grid, grid_pos, ax=ax, edge_color="#cbd5e1", width=1.6)
    drawn = nx.draw_networkx_nodes(
        grid,
        grid_pos,
        ax=ax,
        node_color=values,
        cmap="magma",
        vmin=0,
        vmax=max(signal.max() for signal in signals),
        edgecolors="#0f172a",
        node_size=520,
    )
    nx.draw_networkx_labels(grid, grid_pos, ax=ax, font_size=7, font_color="white")
    ball = [node for node, distance in nx.single_source_shortest_path_length(grid, source_node, cutoff=layer).items()]
    ax.set_title(f"layer {layer}\nreachable nodes: {len(ball)}", fontsize=9)
fig.colorbar(drawn, ax=axes.ravel().tolist(), fraction=0.02, pad=0.01)

message_path = track(
    save_matplotlib(fig, CHAPTER, "message-passing-radius-flow.png", root=COURSE_ARTIFACTS),
    image=True,
)
plt.close(fig)

support_checks = {}
for layer, values in enumerate(signals):
    support = set(np.flatnonzero(values > 1e-12).tolist())
    ball = set(nx.single_source_shortest_path_length(grid, source_node, cutoff=layer).keys())
    support_checks[layer] = support == ball

neighbor_values = np.array([0.2, 1.5, -0.3, 0.9])
shuffled_neighbor_values = neighbor_values[[2, 0, 3, 1]]
checks["message_passing"] = {
    "source_node": source_node,
    "support_equals_distance_ball_each_layer": {str(k): bool(v) for k, v in support_checks.items()},
    "sum_aggregator_order_invariant": bool(np.isclose(neighbor_values.sum(), shuffled_neighbor_values.sum())),
    "mean_aggregator_order_invariant": bool(np.isclose(neighbor_values.mean(), shuffled_neighbor_values.mean())),
    "max_aggregator_order_invariant": bool(np.isclose(neighbor_values.max(), shuffled_neighbor_values.max())),
    "layer_3_total_mass_after_mean_updates": float(signals[3].sum()),
}

display_artifact(message_path, width=880)

## Sanity checks

The final cell verifies the chapter invariants directly. It checks artifact existence and nonblank PNGs, plus the algebraic and graph-theoretic facts used in the explanations: row sums equal degrees, shortest-path distances are symmetric for the undirected example, graph family counts match known formulas, unit-disk edges obey the distance threshold, permutation matrices are orthogonal, the homomorphism preserves edges, Laplacian energy matches edge differences, Fourier reconstruction works, and message-passing support agrees with graph distance.

In [ ]:
assert checks["adjacency_degree"]["tree_adjacency_symmetric"]
assert checks["adjacency_degree"]["tree_row_sums_equal_degrees"]
assert checks["distances_and_families"]["distance_matrix_symmetric"]
assert checks["distances_and_families"]["complete_graph_edge_count_correct"]
assert checks["geometric_graphs"]["unit_disk_all_edges_within_epsilon"]
assert checks["geometric_graphs"]["homophilic_exceeds_heterophilic"]
assert checks["threshold_lab"]["edge_count_monotone"]
assert checks["threshold_lab"]["components_nonincreasing"]
assert checks["permutation"]["orthogonal"]
assert checks["permutation"]["papT_matches_direct_reordered_adjacency"]
assert checks["permutation"]["adjacency_spectrum_invariant"]
assert checks["permutation"]["message_update_equivariance_residual"] < 1e-12
assert checks["homomorphism"]["edge_preserved"]
assert not checks["homomorphism"]["bijective"]
assert checks["laplacian"]["laplacian_symmetric"]
assert checks["laplacian"]["laplacian_psd_min_eigenvalue"] > -1e-10
assert checks["laplacian"]["zero_eigenvalue_multiplicity"] == checks["laplacian"]["connected_components"]
assert checks["laplacian"]["energy_matches_edge_sum_residual"] < 1e-10
assert -1e-10 <= checks["laplacian"]["normalized_spectrum_min"] <= 1e-8
assert checks["laplacian"]["normalized_spectrum_max"] <= 2.0 + 1e-10
assert checks["graph_fourier"]["orthonormal_residual"] < 1e-10
assert checks["graph_fourier"]["reconstruction_residual"] < 1e-10
assert checks["graph_fourier"]["energy_identity_residual"] < 1e-10
assert all(checks["message_passing"]["support_equals_distance_ball_each_layer"].values())

checks["artifact_manifest"] = [
    {"path": book_rel(path), "bytes": int(path.stat().st_size)}
    for path in artifact_paths
    if path.exists()
]
checks_path = track(save_json(json_ready(checks), CHAPTER, "chapter-07-graph-theory-checks.json", root=COURSE_ARTIFACTS))

artifact_records = assert_chapter_artifacts(artifact_paths)
image_records = [assert_nonblank_image(path) for path in image_paths]

print(f"Verified {len(artifact_records)} artifacts.")
print(f"Verified {len(image_records)} nonblank PNG figures.")
print(f"Checks written to {book_rel(checks_path)}")
display_artifact(checks_path)

## Takeaways

- A graph is a discrete geometric domain once its edges define locality and shortest-path distance.
- Adjacency and degree matrices are not just storage formats; they are the algebraic interface for graph operators.
- Geometric graph construction turns metric proximity into edges, but the chosen rule changes connectedness, degree, diameter, and homophily.
- Permutation symmetry separates graph structure from arbitrary node ordering. Pooling should be invariant, while nodewise updates should be equivariant.
- Homomorphisms preserve adjacency without requiring a one-to-one match between vertex sets.
- The graph Laplacian measures signal roughness, detects components, and supplies Fourier modes for graph signals.
- Message passing is local by construction: each layer expands the receptive field by one graph hop.